Тестирование модели, обработка валидационных данных, создание и сохранение предсказаний

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# 1. Загрузка валидационных данных
print("Загрузка валидационных данных за февраль 2025...")
try:
    val_data = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/TVR_prediction/validation_data_022025.xlsx',
                            parse_dates=['researchDate', 'programFirstIssueDate'])
    print(f"Размер данных: {val_data.shape}")
except Exception as e:
    print(f"Ошибка загрузки из Excel: {e}")
    try:
        val_data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/TVR_prediction/validation_data_022025.csv',
                              parse_dates=['researchDate', 'programFirstIssueDate'])
        print(f"Загружено из CSV. Размер: {val_data.shape}")
    except Exception as e2:
        print(f"Также не удалось загрузить из CSV: {e2}")
        raise

# Функция для простой подготовки признаков
def prepare_features_simple(df):
    """Применяет инженерию признаков без использования категориальных типов"""
    print("\nПрименение простой инженерии признаков...")
    df = df.copy()

    # Заполнение пропущенных значений для programAgeRestrictionName
    if 'programAgeRestrictionName' in df.columns:
        if df['programAgeRestrictionName'].isnull().sum() > 0:
            print(f"Заполнение {df['programAgeRestrictionName'].isnull().sum()} пропущенных значений возрастных ограничений значением '16+'")
            df['programAgeRestrictionName'] = df['programAgeRestrictionName'].fillna('16+')

        # Преобразование в числа
        age_mapping = {
            '0+': 0, '3+': 1, '4+': 1, '5+': 1, '6+': 1,
            '12+': 2, '14+': 3, '16+': 3, '18+': 4, '21+': 4
        }
        df['programAgeRestrictionName'] = df['programAgeRestrictionName'].map(age_mapping).fillna(3).astype(float)

    # Преобразование researchDayType в числовой формат
    if 'researchDayType' in df.columns:
        day_type_mapping = {'W': 0, 'E': 1, 'H': 2}
        df['researchDayType'] = df['researchDayType'].map(day_type_mapping).fillna(0).astype(float)

    # Преобразование breaksPrimeTimeStatusName в числовой формат
    if 'breaksPrimeTimeStatusName' in df.columns:
        prime_mapping = {'Прайм-тайм': 1, 'Не прайм-тайм': 0}
        df['breaksPrimeTimeStatusName'] = df['breaksPrimeTimeStatusName'].map(prime_mapping).fillna(0).astype(float)

    # Извлечение признаков даты
    if 'researchDate' in df.columns:
        df['day_of_week'] = df['researchDate'].dt.dayofweek.astype(float)
        df['month'] = df['researchDate'].dt.month.astype(float)


        def get_season(month):
            if month in [12, 1, 2]: return 1.0  # Зима
            elif month in [3, 4, 5]: return 2.0  # Весна
            elif month in [6, 7, 8]: return 3.0  # Лето
            else: return 4.0  # Осень

        df['season'] = df['month'].apply(get_season)

    # Убедимся, что числовые столбцы имеют тип float
    for col in ['breaksDuration', 'programDuration', 'tvCompanyId']:
        if col in df.columns:
            df[col] = df[col].astype(float)


    df['lag_1'] = 0.9883  # Из обучающих данных
    df['lag_5'] = 0.9878  # Из обучающих данных
    df['rolling_mean_5'] = 0.9882  # Из обучающих данных
    df['rolling_mean_10'] = 0.9880  # Из обучающих данных

    return df

# Обработка валидационных данных
val_data_processed = prepare_features_simple(val_data)


features = [
    'researchDayType',
    'programAgeRestrictionName',
    'breaksPrimeTimeStatusName',
    'breaksDuration',
    'programDuration',
    'tvCompanyId',
    'lag_1',
    'lag_5',
    'rolling_mean_5',
    'rolling_mean_10',
    'day_of_week',
    'month',
    'season'
]

# Убедимся, что все признаки существуют
for feature in features:
    if feature not in val_data_processed.columns:
        print(f"Предупреждение: Отсутствует признак {feature}, добавляем значение по умолчанию 0")
        val_data_processed[feature] = 0.0

# Создаем X_val
X_val = val_data_processed[features].copy()

# Убедимся, что все признаки имеют тип float
X_val = X_val.astype(float)

print("\nВыбранные признаки:")
for col in X_val.columns:
    print(f"{col}: {X_val[col].dtype}")

print(f"Размер X_val: {X_val.shape}")

# Загрузка Optuna модели LightGBM
print("\nЗагрузка модели LightGBM Optuna...")


model_paths = [
    "/content/drive/MyDrive/Colab Notebooks/TVR_prediction/TVR_models/lightgbm_optuna_model.pkl",
    "lightgbm_optuna_model.pkl",
    f"{os.getcwd()}/lightgbm_optuna_model.pkl"
]

lgb_model = None
for path in model_paths:
    try:
        lgb_model = joblib.load(path)
        print(f"Загружена модель из: {path}")
        break
    except:
        continue

if lgb_model is None:
    raise FileNotFoundError("Не удалось найти модель LightGBM Optuna")

# Извлечение строки модели
print("Извлечение параметров модели...")
model_str = lgb_model._Booster.model_to_string()

print("Создание нового бустера без информации о категориальных признаках...")
new_booster = lgb.Booster(model_str=model_str)

print("Создание предсказаний...")
raw_preds = new_booster.predict(X_val.values)

# Преобразование обратно из логарифмической шкалы
val_predictions = np.expm1(raw_preds)

print(f"Сгенерировано {len(val_predictions)} предсказаний")
print(f"Статистика предсказаний: мин={val_predictions.min():.4f}, макс={val_predictions.max():.4f}, среднее={val_predictions.mean():.4f}")

# Сохранение предсказаний
val_data['predictions'] = val_predictions

# Сохранение результатов
output_file = '/content/drive/MyDrive/Colab Notebooks/TVR_prediction/february_2025_predictions.csv'
val_data.to_csv(output_file, index=False)
print(f"\nСохранены предсказания в {output_file}")
print("\nГотово!")